In [ ]:
import base64,pathlib,json,traceback,os
import papermill as pm
os.environ['PYTHONHASHSEED']='42'
TASKS={'similarity': '...', 'nli3': '...', 'retrieval': '...', 'temporal': '...'}
report={}
for name,payload in TASKS.items():
    source=pathlib.Path('/kaggle/working')/('_source_'+name+'.ipynb')
    source.write_bytes(base64.b64decode(payload))
    try:
        pm.execute_notebook(str(source),'/kaggle/working/'+name+'_review.ipynb',kernel_name='python3',cwd='/kaggle/working',progress_bar=False,log_output=True)
        report[name]={'status':'complete'}
    except Exception as error:
        traceback.print_exc();report[name]={'status':'error','error':str(error)[-2000:]}
    finally:
        source.unlink(missing_ok=True)
        pathlib.Path('/kaggle/working/suite_status.json').write_text(json.dumps(report,indent=2))
if any(v['status']=='error' for v in report.values()):raise RuntimeError('Sebagian notebook gagal; lihat suite_status.json dan output masing-masing.')
